# Consolidação brasileira em grade completa de cinco minutos

A grade de 2025, no fuso `America/Sao_Paulo`, é a base do merge. Os registros da estação e as somas dos inversores são associados independentemente. A falta de um registro da estação não elimina potência ou energia existente. `datetime` permanece exportado em UTC.

As regras originais de seleção de duplicatas e soma com `min_count=1` foram preservadas. `n_inversores_potencia` distingue a soma completa da soma parcial. Os indicadores de presença identificam a origem das ausências. Nenhum valor ausente é imputado e zero medido permanece zero.

Execute de cima para baixo para gerar `Leopoldo_base_consolidada_5min.csv`. O notebook `notebooks/tcc/03_02_caracterizacao_dados_crus.ipynb` utiliza esse CSV para a análise.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


In [2]:
# Fonte e calendário da usina estudada
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/raw/brasil").is_dir())
pasta = RAIZ / "data/raw/brasil"
USINA = "07-07FX"
ANO = 2025

arquivo_est = pasta / "Leopoldo-fixa-est.csv"
arquivo_inv = pasta / "Leopoldo-fixa-inv.csv"

# Colunas que queremos somar entre os inversores
colunas_inversores = [
    "daily_power_yields_kwh",
    "monthly_power_yields_kwh",
    "total_power_yields_kwh",
    "total_active_power_w",
    "total_dc_power_w"
]

# Estação meteorológica: todas as colunas
df_est = pd.read_csv(arquivo_est, low_memory=False)

# Inversores: apenas as colunas necessárias
df_inv = pd.read_csv(
    arquivo_inv,
    usecols=[
        "datetime",
        "usine_code",
        "meta.id_inversor"
    ] + colunas_inversores,
    low_memory=False
)

df_est = df_est.loc[df_est.usine_code.eq(USINA)].copy()
df_inv = df_inv.loc[df_inv.usine_code.eq(USINA)].copy()

print("Estação:", df_est.shape)
print("Inversores:", df_inv.shape)

Estação: (35535, 64)
Inversores: (733389, 8)


In [3]:
def preparar_horarios(df):
    df = df.copy()
    df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce", utc=True)
    df["data_5min"] = df["datetime"].dt.floor("5min")
    return df.dropna(subset=["data_5min"]).copy()

df_est = preparar_horarios(df_est)
df_inv = preparar_horarios(df_inv)


In [4]:
# Contar os campos preenchidos
df_est["_validos"] = df_est.notna().sum(axis=1)

# Selecionar um registro por intervalo
df_est_5min = (
    df_est
    .sort_values([
        "usine_code",
        "data_5min",
        "_validos",
        "datetime"
    ])
    .drop_duplicates(
        subset=["usine_code", "data_5min"],
        keep="last"
    )
    .drop(columns=["datetime", "_validos"])
    .rename(columns={"data_5min": "datetime"})
    .reset_index(drop=True)
)

df_est_5min["tem_registro_estacao"] = True
print("Estação consolidada:", df_est_5min.shape)

Estação consolidada: (33383, 65)


In [5]:
# Remover registros sem nenhuma medição útil
df_inv = df_inv.dropna(
    subset=colunas_inversores,
    how="all"
).copy()

# Contar medições preenchidas
df_inv["_validos"] = (
    df_inv[colunas_inversores]
    .notna()
    .sum(axis=1)
)

# Manter uma leitura por inversor em cada intervalo
df_inv_unicos = (
    df_inv
    .sort_values([
        "usine_code",
        "meta.id_inversor",
        "data_5min",
        "_validos",
        "datetime"
    ])
    .drop_duplicates(
        subset=[
            "usine_code",
            "meta.id_inversor",
            "data_5min"
        ],
        keep="last"
    )
)

# Somar as medições dos inversores
df_inv_5min = (
    df_inv_unicos
    .groupby(
        ["usine_code", "data_5min"],
        as_index=False
    )[colunas_inversores]
    .sum(min_count=1)
    .rename(columns={"data_5min": "datetime"})
)

# Preserva a soma com min_count=1, mas registra a quantidade de contribuintes.
contagem_inv = (
    df_inv_unicos.groupby(["usine_code", "data_5min"])
    .agg(n_inversores=("meta.id_inversor", "nunique"),
         n_inversores_potencia=("total_active_power_w", "count"))
    .reset_index().rename(columns={"data_5min": "datetime"})
)
df_inv_5min = df_inv_5min.merge(contagem_inv, on=["usine_code", "datetime"], validate="one_to_one")
df_inv_5min["tem_registro_inversores"] = True
print("Inversores consolidados:", df_inv_5min.shape)

display(df_inv_5min.head())

Inversores consolidados: (84235, 10)


,usine_code,datetime,daily_power_yields_kwh,monthly_power_yields_kwh,total_power_yields_kwh,total_active_power_w,total_dc_power_w,n_inversores,n_inversores_potencia,tem_registro_inversores
0,07-07FX,2025-01-02 23:35:00+00:00,722.2,1297.7,169820.0,0.0,0.0,1,1,True
1,07-07FX,2025-01-04 15:20:00+00:00,348.0,2440.4,158509.0,42599.0,43248.0,1,1,True
2,07-07FX,2025-01-04 20:25:00+00:00,515.2,2740.4,181360.0,23922.0,24893.0,1,1,True
3,07-07FX,2025-01-04 21:55:00+00:00,3996.0,21037.9,1060397.0,0.0,0.0,8,8,True
4,07-07FX,2025-01-04 22:05:00+00:00,3996.0,21037.9,1060397.0,0.0,0.0,8,8,True


In [6]:
# A grade completa, em vez da estação, é a base do merge.
# Cada fonte é associada independentemente: faltar meteorologia não apaga potência.
calendario_local = pd.date_range(
    f"{ANO}-01-01", f"{ANO + 1}-01-01", freq="5min",
    inclusive="left", tz="America/Sao_Paulo"
)
grade = pd.DataFrame({"datetime": calendario_local.tz_convert("UTC"), "usine_code": USINA})
df_final = (
    grade.merge(df_est_5min, on=["datetime", "usine_code"], how="left", validate="one_to_one")
    .merge(df_inv_5min, on=["datetime", "usine_code"], how="left", validate="one_to_one")
    .sort_values(["usine_code", "datetime"]).reset_index(drop=True)
)
for coluna in ["tem_registro_estacao", "tem_registro_inversores"]:
    df_final[coluna] = df_final[coluna].eq(True)
for coluna in ["n_inversores", "n_inversores_potencia"]:
    df_final[coluna] = df_final[coluna].fillna(0).astype(int)
print("Dimensões finais:", df_final.shape)
display(df_final.head())


Dimensões finais: (105120, 73)


,datetime,usine_code,meta.id,meta.id_usina,meta.solar_station_id,indice_albedo_medio,gri_kwh_day_est,albedo_tracker,gri_irr_med,poa_kwh_hour,precipitacao_acc,dir_vento,poa_irradiancia,poa_irr_med,air_speed_max,air_temperature_med,indice_albedo_tracker,gri_kwh_acc,poa_irr_max,temperatura_ambiente,real_datetime,poa_kwh_min,tensao_bateria,calculo_albedo_medio,vel_vento,ghi_irr_max,air_temperature_min,ghi_irr_med,ghi_irr_min,ghi_kwh_min,gri_irradiancia,temperatura_painel,indice_albedo_maximo,ghi_kwh_day,gri_irr_min,ghi_kwh_day_est,pv_temperatura_med,poa_irr_min,ghi_kwh_hour,ghi_irradiancia,dir_vento_med,precipitacao_acc_hour,gri_kwh_day,poa_kwh_day_est,poa_kwh_acc,air_temperature_max,ghi_kwh_acc,precipitacao_acc_min,fail_read,poa_kwh_day,pv_temperatura_max,air_speed_med,equipment_name,indice_albedo_minimo,gri_kwh_min,gri_kwh_hour,pv_temperatura_min,air_speed_min,precipitacao_acc_day,poa_kwh_acc_est,gri_irr_max,ghi_kwh_acc_est,_id,gri_kwh_acc_est,tem_registro_estacao,daily_power_yields_kwh,monthly_power_yields_kwh,total_power_yields_kwh,total_active_power_w,total_dc_power_w,n_inversores,n_inversores_potencia,tem_registro_inversores
0,2025-01-01 03:00:00+00:00,07-07FX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,0,0,False
1,2025-01-01 03:05:00+00:00,07-07FX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,0,0,False
2,2025-01-01 03:10:00+00:00,07-07FX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,0,0,False
3,2025-01-01 03:15:00+00:00,07-07FX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,0,0,False
4,2025-01-01 03:20:00+00:00,07-07FX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,0,0,False


In [7]:
# Contagens e conferência de que nenhuma potência existente foi perdida no merge.
print("Total de intervalos:", len(df_final))
print("Com potência:", int(df_final.total_active_power_w.notna().sum()))
print("Sem potência:", int(df_final.total_active_power_w.isna().sum()))
print("Com potência e sem estação:", int((df_final.total_active_power_w.notna() & ~df_final.tem_registro_estacao).sum()))
print("Com estação:", int(df_final.tem_registro_estacao.sum()))
assert not df_final.duplicated(["usine_code", "datetime"]).any()
assert len(df_final) == len(calendario_local)
assert df_final.datetime.diff().dropna().eq(pd.Timedelta(minutes=5)).all()
esperado = grade.merge(df_inv_5min, on=["usine_code", "datetime"], how="left", validate="one_to_one")
for coluna in colunas_inversores:
    np.testing.assert_allclose(df_final[coluna], esperado[coluna], equal_nan=True)
print("Conferido: potências e energias preservadas em todos os horários da grade.")


Total de intervalos: 105120
Com potência: 84235
Sem potência: 20885
Com potência e sem estação: 52683
Com estação: 33383
Conferido: potências e energias preservadas em todos os horários da grade.


In [8]:
# A soma continua admitindo parcelas disponíveis, como no notebook original.
# A contagem permite identificar as somas com oito inversores ou apenas parte deles.
print("Quantidade de inversores com potência por intervalo:")
print(df_final.n_inversores_potencia.value_counts().sort_index())
print("Potência dos oito inversores:", int(df_final.n_inversores_potencia.eq(8).sum()))
print("Potência parcial:", int(df_final.n_inversores_potencia.between(1, 7).sum()))
display(df_final.loc[df_final.total_active_power_w.notna() & ~df_final.tem_registro_estacao,
    ["datetime", "total_active_power_w", "daily_power_yields_kwh", "n_inversores_potencia", "air_temperature_med", "ghi_irr_med"]].head(12))


Quantidade de inversores com potência por intervalo:
n_inversores_potencia
0    20885
1      170
2      263
3      264
4      260
5      196
6      105
7      190
8    82787
Name: count, dtype: int64
Potência dos oito inversores: 82787
Potência parcial: 1448


,datetime,total_active_power_w,daily_power_yields_kwh,n_inversores_potencia,air_temperature_med,ghi_irr_med
535,2025-01-02 23:35:00+00:00,0.0,722.2,1,NaN,NaN
1012,2025-01-04 15:20:00+00:00,42599.0,348.0,1,NaN,NaN
1073,2025-01-04 20:25:00+00:00,23922.0,515.2,1,NaN,NaN
1091,2025-01-04 21:55:00+00:00,0.0,3996.0,8,NaN,NaN
1562,2025-01-06 13:10:00+00:00,41470.0,194.5,1,NaN,NaN
1606,2025-01-06 16:50:00+00:00,295088.0,2085.5,4,NaN,NaN
2011,2025-01-08 02:35:00+00:00,0.0,725.8,1,NaN,NaN
3573,2025-01-13 12:45:00+00:00,60756.0,123.1,1,NaN,NaN
3619,2025-01-13 16:35:00+00:00,88818.0,402.3,1,NaN,NaN
3622,2025-01-13 16:50:00+00:00,125962.0,458.8,1,NaN,NaN


In [9]:
ARQUIVO_SAIDA = RAIZ / "Leopoldo_base_consolidada_5min.csv"
df_final.to_csv(ARQUIVO_SAIDA, index=False)
print("Arquivo salvo:", ARQUIVO_SAIDA)
print("Calendário de 2025 no horário local; datetime exportado em UTC.")


Arquivo salvo: C:\Users\camil\Documents\transfer-learning-energy-forecast-tcc\Leopoldo_base_consolidada_5min.csv
Calendário de 2025 no horário local; datetime exportado em UTC.


In [10]:
df_final.columns

Out[10]: 
Index(['datetime', 'usine_code', 'meta.id', 'meta.id_usina',
       'meta.solar_station_id', 'indice_albedo_medio', 'gri_kwh_day_est',
       'albedo_tracker', 'gri_irr_med', 'poa_kwh_hour', 'precipitacao_acc',
       'dir_vento', 'poa_irradiancia', 'poa_irr_med', 'air_speed_max',
       'air_temperature_med', 'indice_albedo_tracker', 'gri_kwh_acc',
       'poa_irr_max', 'temperatura_ambiente', 'real_datetime', 'poa_kwh_min',
       'tensao_bateria', 'calculo_albedo_medio', 'vel_vento', 'ghi_irr_max',
       'air_temperature_min', 'ghi_irr_med', 'ghi_irr_min', 'ghi_kwh_min',
       'gri_irradiancia', 'temperatura_painel', 'indice_albedo_maximo',
       'ghi_kwh_day', 'gri_irr_min', 'ghi_kwh_day_est', 'pv_temperatura_med',
       'poa_irr_min', 'ghi_kwh_hour', 'ghi_irradiancia', 'dir_vento_med',
       'precipitacao_acc_hour', 'gri_kwh_day', 'poa_kwh_day_est',
       'poa_kwh_acc', 'air_temperature_max', 'ghi_kwh_acc',
       'precipitacao_acc_min', 'fail_read', 'poa_kwh_day',

## Dicionário investigado de todas as colunas

Esta seção explica cada uma das 73 colunas do CSV final e as três colunas auxiliares usadas na consolidação. A investigação combina o código acima, o esquema original, relações numéricas no arquivo e referências técnicas gerais. Não foi localizado um manual do fornecedor com esses nomes de campo; por isso, definições nominais, inferências e pendências aparecem explicitamente.

* **Potência elétrica:** W; para apresentar em kW, dividir por 1000.
* **Energia elétrica:** kWh nos contadores dos inversores. As somas são estados dos contadores no horário, sem integração temporal dos cinco minutos.
* **Irradiância solar:** fluxo de potência por área, usualmente W/m². **Irradiação:** energia por área, usualmente kWh/m². A escala de cada sensor local ainda exige documentação.
* **GHI:** irradiância global horizontal. **POA:** irradiância no plano dos módulos. **GRI:** acepção técnica usual de irradiância refletida pelo solo; a associação ao sensor local precisa de confirmação.
* **Sufixo min:** em `*_irr_min`, indica mínimo; em `*_kwh_min`, os dados são fortemente compatíveis com **um minuto**, não com mínimo e não com a energia integral dos cinco minutos.
* **Sufixo est:** sua expansão não está documentada. Várias dessas colunas repetem a versão sem est no arquivo; não devem ser tratadas automaticamente como previsões ou estimativas independentes.

A grade final de cinco minutos não define, por si só, a janela original de média, máximo, mínimo ou integração dos sensores. O dicionário apenas documenta as colunas e não altera as medições ou a pipeline.


In [11]:
import sys
from IPython.display import display, Markdown
if str(RAIZ / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ / "src"))
from energia_transfer.dicionario_leopoldo import investigar_e_exportar, REFERENCIAS

PASTA_DICIONARIO = RAIZ / "reports/tcc/dicionario_colunas_leopoldo"
investigacao_colunas = investigar_e_exportar(df_final, PASTA_DICIONARIO)
dicionario_colunas = investigacao_colunas["dicionario"]
assert dicionario_colunas.coluna.tolist() == df_final.columns.tolist()
assert dicionario_colunas.coluna.is_unique
campos_explicacao = ["coluna", "significado_pt", "unidade", "interpretacao_temporal", "fundamento", "confirmacao_pendente"]
with pd.option_context("display.max_rows", None, "display.max_colwidth", None, "display.max_columns", None):
    for grupo in dicionario_colunas.grupo.unique():
        display(Markdown("### " + grupo))
        display(dicionario_colunas.loc[dicionario_colunas.grupo.eq(grupo), campos_explicacao].reset_index(drop=True))
print("Colunas finais explicadas:", len(dicionario_colunas))
print("Dicionário completo com origem, cobertura e exemplos:", PASTA_DICIONARIO / "dicionario_colunas.csv")


Colunas finais explicadas: 73
Dicionário completo com origem, cobertura e exemplos: C:\Users\camil\Documents\transfer-learning-energy-forecast-tcc\reports\tcc\dicionario_colunas_leopoldo\dicionario_colunas.csv


### Tempo e identificação

,coluna,significado_pt,unidade,interpretacao_temporal,fundamento,confirmacao_pendente
0,datetime,Início do intervalo de cinco minutos usado como chave do merge,data/hora UTC,UTC no CSV; a grade de 2025 é construída no fuso America/Sao_Paulo e convertida para UTC,Confirmado pelo código de data.ipynb,
1,usine_code,Código da usina selecionada: 07-07FX,identificador,Constante em todos os intervalos da grade,Confirmado pelo código de data.ipynb,
2,meta.id,Identificador interno do equipamento/registro de estação nos metadados,identificador,"Atributo do registro da estação, não uma medição temporal",Origem confirmada pelo código; interpretação nominal pelo esquema,Esquema dos identificadores do fornecedor; NaN sem estação é esperado
3,meta.id_usina,Identificador interno da usina nos metadados da estação,identificador,"Atributo do registro da estação, não uma medição temporal",Origem confirmada pelo código; interpretação nominal pelo esquema,Esquema dos identificadores do fornecedor; NaN sem estação é esperado
4,meta.solar_station_id,Identificador interno da estação solarimétrica,identificador,"Atributo do registro da estação, não uma medição temporal",Origem confirmada pelo código; interpretação nominal pelo esquema,Esquema dos identificadores do fornecedor; NaN sem estação é esperado
5,real_datetime,"Horário adicional fornecido pela estação, preservado no formato original",data/hora; valores observados em UTC,Não é a chave de cinco minutos; pode diferir de datetime,Nome e formato observados no CSV,"Confirmar se representa aquisição, relógio do equipamento, transmissão ou recepção"
6,equipment_name,"Nome do equipamento da estação; neste arquivo, est_sols1",identificador,"Atributo do registro da estação, não uma medição temporal",Origem confirmada pelo código; interpretação nominal pelo esquema,Esquema dos identificadores do fornecedor; NaN sem estação é esperado
7,_id,Identificador do registro original selecionado da estação,identificador,"Atributo do registro da estação, não uma medição temporal",Origem confirmada pelo código; interpretação nominal pelo esquema,Esquema dos identificadores do fornecedor; NaN sem estação é esperado


### Albedo

,coluna,significado_pt,unidade,interpretacao_temporal,fundamento,confirmacao_pendente
0,indice_albedo_medio,"Índice de albedo médio, pelo nome do campo",não confirmada: razão ou %,Janela da estatística e método de cálculo não documentados,Conceito documentado; mapeamento local pelo nome e evidência numérica,Confirmar fórmula e sensores; tracker no nome não comprova que a usina tenha rastreamento. Índices vazios não permitem confirmar a escala
1,albedo_tracker,"Albedo rotulado como tracker: razão de radiação refletida e incidente, segundo a nomenclatura","razão adimensional, nominal",Janela da estatística e método de cálculo não documentados,Conceito documentado; mapeamento local pelo nome e evidência numérica,Confirmar fórmula e sensores; tracker no nome não comprova que a usina tenha rastreamento. Índices vazios não permitem confirmar a escala
2,indice_albedo_tracker,"Índice de albedo rotulado como tracker, com escala aproximadamente percentual nos valores observados",% sugerido pelos dados; confirmar,Janela da estatística e método de cálculo não documentados,Conceito documentado; mapeamento local pelo nome e evidência numérica,Confirmar fórmula e sensores; tracker no nome não comprova que a usina tenha rastreamento. Índices vazios não permitem confirmar a escala
3,calculo_albedo_medio,"Campo de cálculo do albedo médio, pelo nome original",não confirmada: razão ou %,Janela da estatística e método de cálculo não documentados,Conceito documentado; mapeamento local pelo nome e evidência numérica,Confirmar fórmula e sensores; tracker no nome não comprova que a usina tenha rastreamento. Índices vazios não permitem confirmar a escala
4,indice_albedo_maximo,"Índice de albedo máximo, pelo nome do campo",não confirmada: razão ou %,Janela da estatística e método de cálculo não documentados,Conceito documentado; mapeamento local pelo nome e evidência numérica,Confirmar fórmula e sensores; tracker no nome não comprova que a usina tenha rastreamento. Índices vazios não permitem confirmar a escala
5,indice_albedo_minimo,"Índice de albedo mínimo, pelo nome do campo",não confirmada: razão ou %,Janela da estatística e método de cálculo não documentados,Conceito documentado; mapeamento local pelo nome e evidência numérica,Confirmar fórmula e sensores; tracker no nome não comprova que a usina tenha rastreamento. Índices vazios não permitem confirmar a escala


...

,coluna,significado_pt,unidade,interpretacao_temporal,fundamento,confirmacao_pendente
0,fail_read,"Indicador original de falha de leitura, pelo nome do campo",booleano/código,Associado ao registro da estação,Interpretação pelo nome; distribuição dos valores verificada no arquivo,Confirmar codificação com o fornecedor; False não comprova calibração ou qualidade física


### Indicadores de disponibilidade

,coluna,significado_pt,unidade,interpretacao_temporal,fundamento,confirmacao_pendente
0,tem_registro_estacao,Indica que existe uma linha da estação selecionada naquele intervalo,booleano,Calculado separadamente para cada intervalo de cinco minutos,Confirmado pelo código de data.ipynb,Não exige que todos os canais físicos estejam preenchidos; True não significa leitura validada
1,n_inversores,Quantidade de identificadores distintos de inversor com alguma medição útil após deduplicação,contagem de 0 a 8,Calculado separadamente para cada intervalo de cinco minutos,Confirmado pelo código de data.ipynb,Não exige que a potência ou todos os contadores estejam preenchidos
2,n_inversores_potencia,Quantidade de inversores com potência AC não ausente naquele intervalo,contagem de 0 a 8,Calculado separadamente para cada intervalo de cinco minutos,Confirmado pelo código de data.ipynb,Oito indica soma com os oito equipamentos; de um a sete indica potência parcial; zero não indica geração nula
3,tem_registro_inversores,Indica que existe um grupo de inversores com pelo menos uma das cinco medições úteis naquele intervalo,booleano,Calculado separadamente para cada intervalo de cinco minutos,Confirmado pelo código de data.ipynb,Grupos sem todas as cinco medições são descartados antes deste indicador; não equivale à existência de qualquer linha bruta


### Potência e energia dos inversores

,coluna,significado_pt,unidade,interpretacao_temporal,fundamento,confirmacao_pendente
0,daily_power_yields_kwh,"Energia elétrica do contador diário, somada entre os inversores disponíveis",kWh,Estado do contador diário registrado no horário; não é energia dos cinco minutos,Agregação confirmada pelo código; unidade indicada pelo nome original,"A soma pode ser parcial; n_inversores_potencia conta contribuintes da potência AC, não de cada contador. Validação física dos contadores é outra etapa"
1,monthly_power_yields_kwh,"Energia elétrica do contador mensal, somada entre os inversores disponíveis",kWh,Estado do contador mensal registrado no horário; não somar novamente ao longo dos horários,Agregação confirmada pelo código; unidade indicada pelo nome original,"A soma pode ser parcial; n_inversores_potencia conta contribuintes da potência AC, não de cada contador. Validação física dos contadores é outra etapa"
2,total_power_yields_kwh,Energia elétrica acumulada dos contadores totais dos inversores disponíveis,kWh,Estado dos contadores acumulados; horizonte e reinicializações dependem do equipamento,Agregação confirmada pelo código; unidade indicada pelo nome original,"A soma pode ser parcial; n_inversores_potencia conta contribuintes da potência AC, não de cada contador. Validação física dos contadores é outra etapa"
3,total_active_power_w,Potência ativa AC agregada dos inversores disponíveis,W,Leitura de potência no horário; não é energia,Agregação confirmada pelo código; unidade indicada pelo nome original,"A soma pode ser parcial; n_inversores_potencia conta contribuintes da potência AC, não de cada contador. Validação física dos contadores é outra etapa"
4,total_dc_power_w,Potência DC agregada dos inversores disponíveis,W,Leitura de potência em corrente contínua no horário,Agregação confirmada pelo código; unidade indicada pelo nome original,"A soma pode ser parcial; n_inversores_potencia conta contribuintes da potência AC, não de cada contador. Validação física dos contadores é outra etapa"


### Colunas auxiliares e evidências encontradas

`meta.id_inversor`, `data_5min` e `_validos` são usados durante a preparação, mas não são colunas do CSV final. `_validos` prioriza duplicatas por preenchimento e não é um selo de qualidade física.

As verificações seguintes testam relações entre os campos. Elas apoiam interpretações, sem substituir a ficha do sensor ou estabelecer calibração. Para um minuto, W/m² dividido por 60000 resulta em kWh/m²; para cinco minutos, o divisor seria 12000. Os testes desconsideram valores de irradiância com módulo menor ou igual a 10 para evitar conclusões baseadas apenas em zeros e ruído.

Campos diários repetidos em vários horários não comprovam que o total estava disponível durante o próprio dia. Os contadores horários/diários e os campos acumulados precisam de confirmação sobre fechamento, reinicialização e fuso.


In [12]:
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(investigacao_colunas["auxiliares"])
    display(investigacao_colunas["evidencias"])
    display(dicionario_colunas[["coluna", "preenchidos", "cobertura_grade_pct", "valores_distintos", "exemplo", "observacoes_dados"]])


,coluna,significado_pt,origem,destino
0,meta.id_inversor,Identificador do inversor usado na deduplicação e na contagem de equipamentos,CSV original dos inversores,Não é exportado no CSV agregado
1,data_5min,datetime original em UTC associado ao início do intervalo por floor de cinco minutos,Criado em ambas as fontes,Renomeado para datetime na consolidação
2,_validos,Contagem para priorizar duplicatas: na estação conta todas as colunas preenchidas; nos inversores somente as cinco medições selecionadas,Criado pelo notebook,Não é exportado; não representa uma validação física


,tipo,campos,pares,resultado,limite
0,Janela de um minuto,ghi_kwh_min; ghi_irr_med,14333,99.53% compatíveis com kWh_min = irradiância média / 60000,"Filtro |irr_med| > 10; rtol=1e-4, atol=0,05 W/m²; evidência numérica, não calibração."
1,Temporalidade diária,ghi_kwh_day,140,114 dias têm um único valor diário repetido; 26 têm mais de um,Valor repetido não comprova que o total estava disponível naquele horário; fechamento e fuso do contador precisam de confirmação.
2,Variante est,ghi_kwh_day; ghi_kwh_day_est,33383,100.00% de pares numericamente iguais,Igualdade neste arquivo não documenta o significado de est nem demonstra previsão independente.
3,Variante est,ghi_kwh_acc; ghi_kwh_acc_est,33383,100.00% de pares numericamente iguais,Igualdade neste arquivo não documenta o significado de est nem demonstra previsão independente.
4,Janela de um minuto,poa_kwh_min; poa_irr_med,16624,99.39% compatíveis com kWh_min = irradiância média / 60000,"Filtro |irr_med| > 10; rtol=1e-4, atol=0,05 W/m²; evidência numérica, não calibração."
5,Temporalidade diária,poa_kwh_day,140,114 dias têm um único valor diário repetido; 26 têm mais de um,Valor repetido não comprova que o total estava disponível naquele horário; fechamento e fuso do contador precisam de confirmação.
6,Variante est,poa_kwh_day; poa_kwh_day_est,33383,100.00% de pares numericamente iguais,Igualdade neste arquivo não documenta o significado de est nem demonstra previsão independente.
7,Variante est,poa_kwh_acc; poa_kwh_acc_est,33383,100.00% de pares numericamente iguais,Igualdade neste arquivo não documenta o significado de est nem demonstra previsão independente.
8,Janela de um minuto,gri_kwh_min; gri_irr_med,5,100.00% compatíveis com kWh_min = irradiância média / 60000,"Filtro |irr_med| > 10; rtol=1e-4, atol=0,05 W/m²; evidência numérica, não calibração."
9,Temporalidade diária,gri_kwh_day,15,15 dias têm um único valor diário repetido; 0 têm mais de um,Valor repetido não comprova que o total estava disponível naquele horário; fechamento e fuso do contador precisam de confirmação.


,coluna,preenchidos,cobertura_grade_pct,valores_distintos,exemplo,observacoes_dados
0,datetime,105120,100.000000,105120,2025-01-01 03:00:00+00:00,
1,usine_code,105120,100.000000,1,07-07FX,Campo constante entre os valores presentes: 07-07FX.
2,meta.id,33383,31.757040,1,1.0,Campo constante entre os valores presentes: 1.0.
3,meta.id_usina,33383,31.757040,1,50.0,Campo constante entre os valores presentes: 50.0.
4,meta.solar_station_id,33321,31.698059,1,89.0,Campo constante entre os valores presentes: 89.0.
5,indice_albedo_medio,0,0.000000,0,(sem valores),Coluna totalmente vazia; significado e unidade específicos não podem ser validados pelos dados.
6,gri_kwh_day_est,62,0.058980,7,0.0,62 pares disponíveis; 100.00% iguais à coluna sem est (np.isclose padrão).
7,albedo_tracker,62,0.058980,7,0.0,
8,gri_irr_med,62,0.058980,7,0.0,
9,poa_kwh_hour,33383,31.757040,2848,0.7857803106307983,


### Referências e confirmações pendentes

As fontes abaixo sustentam os conceitos físicos e a semântica da soma no pandas. Elas **não são o manual do fornecedor** da usina. Permanecem pendentes: o significado exato de est, a janela das estatísticas, o início e o reset de acumulados, a posição/calibração dos sensores, as escalas de radiação e o significado operacional de real_datetime e fail_read.

As colunas de direção do vento são constantes em 360 no arquivo atual; isso precisa de verificação antes de interpretar como vento permanentemente de norte. Albedos negativos e velocidades negativas também exigem inspeção, sem correção automática nesta etapa.


In [13]:
for sigla, (titulo, url) in REFERENCIAS.items():
    display(Markdown(f"* **{sigla}:** [{titulo}]({url})"))
print("Versão para consulta/reda??o:", PASTA_DICIONARIO / "dicionario_colunas.md")


Versão para consulta/reda??o: C:\Users\camil\Documents\transfer-learning-energy-forecast-tcc\reports\tcc\dicionario_colunas_leopoldo\dicionario_colunas.md


* **NREL:** [Guia de dados de albedo: definições de GHI, GRI, vento e precipitação](https://datahub.duramat.org/dataset/d1c1fc62-eb76-477e-a07b-6bb0dc8e813c/resource/6206db10-c5b4-4635-a870-ec23efd1c181/download/users-guide-for-albedo-data-sets-rev3.pdf)

* **SANDIA:** [Sandia PVPMC: irradiância no plano dos módulos (POA)](https://pvpmc.sandia.gov/modeling-guide/1-weather-design-inputs/plane-of-array-poa-irradiance/)

* **SANDIA_RTC:** [Sandia: instrumentação meteorológica e temperatura dos módulos](https://rtc.sandia.gov/technical-capabilities/)

* **PVSYST:** [PVsyst: irradiância em W/m² e irradiação em kWh/m²](https://www.pvsyst.com/help-pvsyst7/models_meteo_incidentmodels.htm)

* **ALBEDO:** [Kipp & Zonen: definições de albedo e irradiância](https://www.kippzonen.com/Download/64/Manual-Albedometers-English)

* **PANDAS:** [Pandas: soma por grupo e parâmetro min_count](https://pandas.pydata.org/pandas-docs/version/2.0/reference/api/pandas.core.groupby.DataFrameGroupBy.sum.html)